# Task 2: supply-chain QLoRA and matched evaluation

See `specs/task2-genai.md` for the input/output contract and all hyperparameter justifications. This notebook is unexecuted until run on a CUDA Colab GPU. Do not claim training/evaluation before outputs exist.

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile

if not Path("pyproject.toml").exists():
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Run locally from repository root or upload assessment_code.zip in Colab")
    print("Upload assessment_code.zip from the project's dist folder.")
    uploads = files.upload()
    bundle = next((name for name in uploads if name.endswith(".zip")), None)
    if bundle is None:
        raise RuntimeError("Code bundle ZIP required")
    target = Path("/content/cdazzdev_assessment").resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle) as archive:
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if not destination.is_relative_to(target):
                raise ValueError("Unsafe ZIP member")
        archive.extractall(target)
    os.chdir(target)
from importlib.metadata import version, PackageNotFoundError
try:
    version("cdazzdev-mle-assessment")
except PackageNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
print("Workspace:", Path.cwd())

In [ ]:
import os
try:
    from google.colab import userdata
    for name in ["GROQ_API_KEY", "HF_TOKEN"]:
        try:
            value = userdata.get(name)
            if value:
                os.environ[name] = value
        except Exception:
            print(name, "not configured or notebook access disabled")
except ImportError:
    print("Local run uses environment variables.")
from shared.preflight import check
print(check(inference=bool(os.getenv("GROQ_API_KEY"))))

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training]"])
import torch
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
from importlib.metadata import version
print({name:version(name) for name in ["transformers","peft","bitsandbytes","accelerate"]})

In [ ]:
# Persist progress outside the ephemeral runtime.
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/CDAZZDEV_MLE/task2")
except ImportError:
    ARTIFACT_ROOT = Path("task2_genai")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = ARTIFACT_ROOT / "data"
CHECKPOINT_DIR = ARTIFACT_ROOT / "checkpoints"
MERGED_DIR = ARTIFACT_ROOT / "models/merged"
OUTPUT_DIR = ARTIFACT_ROOT / "outputs"
MODEL_REPO_ID = ""  # Set your HF username/model-name only when ready to publish.
RUN_TRAINING = True
RUN_EVALUATION = True
PUBLISH_MODEL = False

## NF4 feasibility gate
This is one forward/backward test; it is not full fine-tuning evidence.

In [ ]:
from task2_genai.train import TrainingConfig, RATIONALE, smoke_test
from dataclasses import asdict
config = TrainingConfig()
print(asdict(config))
print(RATIONALE)
print(smoke_test(config))

## Teacher dataset and explicit oracle
Teacher and student differ. Generation is resumable, validates labels/evidence against supplied policy, and saves the exact teacher system prompt.

In [ ]:
from task2_genai.dataset import generate
from task2_genai.domain import TEACHER_SYSTEM, STUDENT_SYSTEM
print("Teacher system prompt:\n", TEACHER_SYSTEM)
print("Student system prompt:\n", STUDENT_SYSTEM)
diversity = generate(DATA_DIR, count=200)
print(diversity)
assert not diversity["near_duplicate_cross_split_flags"], "Resolve near duplicates before training"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for split, info in diversity["splits"].items():
    axes[0].hist(info["prompt_char_lengths"], alpha=.5, label=split)
axes[0].set_title("Prompt character lengths"); axes[0].legend()
pd.DataFrame({split:info["topic_counts"] for split,info in diversity["splits"].items()}).fillna(0).plot.bar(ax=axes[1])
axes[1].set_title("Topic frequencies"); plt.tight_layout(); plt.show()
print("Keyword frequencies:", diversity["keyword_counts"])
print("Split sizes:", {split:info["count"] for split,info in diversity["splits"].items()})

## Actual QLoRA training
Base weights are frozen NF4, adapters receive assistant-only loss, validation controls checkpoint selection. Inspect all real losses; do not fabricate a decreasing curve.

In [ ]:
from task2_genai.train import train
if RUN_TRAINING:
    history = train(DATA_DIR, CHECKPOINT_DIR, config)
    display(pd.DataFrame(history))
    pd.DataFrame(history).plot(x="epoch", y=["train_loss","validation_loss"], marker="o")
    plt.show()
else:
    print("Training skipped; execution evidence incomplete")

## Merge and reload
The non-quantized base is reloaded for `merge_and_unload()`. Evaluation below reloads the saved merged weights and runs inference.

In [ ]:
from task2_genai.train import merge
if RUN_TRAINING:
    print("Saved merged model:", merge(CHECKPOINT_DIR / "best_adapter", MERGED_DIR, config))

## Frozen test-set baseline comparison
Both models receive the same system prompt, chat inputs, deterministic generation settings, and held-out examples. ROUGE-L and BERTScore are supplemented by schema/category/severity checks.

In [ ]:
from task2_genai.evaluate import evaluate
if RUN_EVALUATION:
    comparison = evaluate(DATA_DIR, MERGED_DIR, OUTPUT_DIR, config.model_id)
    display(pd.DataFrame(comparison).T)
    print((OUTPUT_DIR / "comparison.md").read_text())
else:
    print("Evaluation skipped; execution evidence incomplete")

## Personal response review
Open `manual_review.csv` in Drive. Personally label at least ten rows `correct`, `partially_correct`, or `hallucinated`, and write reasons. Then rerun the next cell. An unfilled review does not count. Write the two analysis paragraphs in `qualitative_analysis.md`.

In [ ]:
from task2_genai.evaluate import summarize_review
try:
    print(summarize_review(OUTPUT_DIR / "manual_review.csv"))
except ValueError as error:
    print("PENDING PERSONAL REVIEW:", error)

## Model card and optional publication step
Set the repository ID and enable publication only after inspecting the model and results. No credentials appear in the notebook.

In [ ]:
from task2_genai.evaluate import publish
import shutil
if MERGED_DIR.exists():
    shutil.copyfile("task2_genai/MODEL_CARD_TEMPLATE.md", MERGED_DIR / "README.md")
if PUBLISH_MODEL:
    assert MODEL_REPO_ID and "/" in MODEL_REPO_ID, "Set your actual HF repository ID"
    print(publish(MERGED_DIR, MODEL_REPO_ID))
else:
    print("Not published. Fill the model card with actual results before uploading.")

## Download submission evidence
Copy data/outputs and relevant training logs from Drive into the GitHub task folder. Keep merged weights on HF/Drive. Download this executed notebook with its outputs. Final status requires real results and the personal review.